# Ejecucion de Ordenes y Costes

Implementacion de la simulacion de ordenes con reglas de ejecucion y costes:
- Ventas a OPEN en el dia de rebalanceo
- Compras a CLOSE en el mismo dia
- Comision 0.23% con minimo 23 USD por orden
- Si un activo deja de cotizar, se vende a CLOSE del ultimo dia disponible y queda en liquidez


In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 50)

In [2]:
# Cargar seleccion de activos (20 symbols por fecha)
selection = pd.read_csv('seleccion_momentum.csv')
selection['date'] = pd.to_datetime(selection['date'])

required_cols = {'date', 'symbol', 'sector', 'weight', 'open', 'close'}
missing_cols = required_cols - set(selection.columns)
if missing_cols:
    raise ValueError(f'Faltan columnas en seleccion_momentum.csv: {sorted(missing_cols)}')

selection = selection.dropna(subset=['date', 'symbol', 'open', 'close', 'weight']).copy()
selection['open'] = pd.to_numeric(selection['open'], errors='coerce')
selection['close'] = pd.to_numeric(selection['close'], errors='coerce')
selection['weight'] = pd.to_numeric(selection['weight'], errors='coerce')
selection = selection.dropna(subset=['open', 'close', 'weight'])

# Garantizar un registro por fecha y symbol
selection = selection.sort_values(['date', 'score'], ascending=[True, False]).drop_duplicates(['date', 'symbol'])

counts = selection.groupby('date')['symbol'].size()
print('Fechas:', selection['date'].nunique())
print('Activos por fecha (min/max):', counts.min(), '/', counts.max())
selection.head()

Fechas: 124
Activos por fecha (min/max): 20 / 20


,date,rank,sector,symbol,score,weight,open,close
0,2015-10-31,1,Industrials,BLDR,2.581801,0.05,11.670,11.820
1,2015-10-31,2,Health Care,ABMD-202212,2.419179,0.05,72.990,73.660
2,2015-10-31,3,Consumer Discretionary,CZR,2.366657,0.05,9.910,9.900
3,2015-10-31,4,Communication Services,CVC-201606,2.229772,0.05,32.560,32.590
4,2015-10-31,5,Communication Services,NFLX,1.944029,0.05,10.512,10.838


In [ ]:
# Precios de mercado diarios para ejecucion (universo completo)
market = pd.read_pickle('sp500_history_filtered.pkl')[['date', 'symbol', 'open', 'close', 'sector']].copy()
market['date'] = pd.to_datetime(market['date'])
market['open'] = pd.to_numeric(market['open'], errors='coerce')
market['close'] = pd.to_numeric(market['close'], errors='coerce')
market = market.dropna(subset=['date', 'symbol', 'open', 'close'])

# Un registro por date+symbol
market = market.sort_values(['date', 'symbol']).drop_duplicates(['date', 'symbol'], keep='last')

print('Rango precios mercado:', market['date'].min().date(), '->', market['date'].max().date())
print('Symbols con precio:', market['symbol'].nunique())
market.head()

Rango precios mercado: 2013-01-02 -> 2026-01-30
Symbols con precio: 860


,date,symbol,open,close,sector
3299,2013-01-02,A,27.011116,26.825365,Health Care
10798,2013-01-02,AABA-201910,5.506497,5.473785,Consumer Discretionary
23155,2013-01-02,AAMRQ-201312,0.800000,0.860000,Industrials
26181,2013-01-02,AAP,63.962387,63.092033,Consumer Discretionary
35268,2013-01-02,AAPL,16.757126,16.612194,Information Technology


In [4]:
# Estructuras auxiliares para consultas rapidas de precios por symbol
prices_by_symbol = {}
for sym, g in market.groupby('symbol', sort=False):
    prices_by_symbol[sym] = g[['date', 'open', 'close']].set_index('date').sort_index()

all_prices = market[['date', 'symbol', 'open', 'close']].copy()
all_prices = all_prices.sort_values(['symbol', 'date'])
all_prices.head()

,date,symbol,open,close
3299,2013-01-02,A,27.011116,26.825365
3300,2013-01-03,A,26.863794,26.921442
3301,2013-01-04,A,26.991901,27.453083
3302,2013-01-07,A,27.286543,27.254519
3303,2013-01-08,A,27.203276,27.036737


In [5]:
# Mapear fecha de rebalanceo (etiqueta fin de mes) a ultimo dia habil real de mercado
selection['rebalance_date'] = pd.to_datetime(selection['date'])

last_trade_by_month = (
    market.groupby(market['date'].dt.to_period('M'))['date']
          .max()
)

selection['trade_date'] = selection['rebalance_date'].dt.to_period('M').map(last_trade_by_month)
selection = selection.dropna(subset=['trade_date']).copy()
selection['trade_date'] = pd.to_datetime(selection['trade_date'])

missing_trade_date = selection['trade_date'].isna().sum()
if missing_trade_date > 0:
    raise ValueError(f'Hay rebalanceos sin trade_date mapeada: {missing_trade_date}')

print('Primer rebalanceo (label/exec):', selection['rebalance_date'].min().date(), '/', selection['trade_date'].min().date())
print('Ultimo rebalanceo (label/exec):', selection['rebalance_date'].max().date(), '/', selection['trade_date'].max().date())
selection.head(20)

Primer rebalanceo (label/exec): 2015-10-31 / 2015-10-30
Ultimo rebalanceo (label/exec): 2026-01-31 / 2026-01-30


,date,rank,sector,symbol,score,weight,open,close,rebalance_date,trade_date
0,2015-10-31,1,Industrials,BLDR,2.581801,0.05,11.670000,11.820000,2015-10-31,2015-10-30
1,2015-10-31,2,Health Care,ABMD-202212,2.419179,0.05,72.990000,73.660000,2015-10-31,2015-10-30
2,2015-10-31,3,Consumer Discretionary,CZR,2.366657,0.05,9.910000,9.900000,2015-10-31,2015-10-30
3,2015-10-31,4,Communication Services,CVC-201606,2.229772,0.05,32.560000,32.590000,2015-10-31,2015-10-30
4,2015-10-31,5,Communication Services,NFLX,1.944029,0.05,10.512000,10.838000,2015-10-31,2015-10-30
5,2015-10-31,6,Health Care,DXCM,1.855606,0.05,20.660000,20.830000,2015-10-31,2015-10-30
6,2015-10-31,7,Industrials,FIX,1.681483,0.05,29.143095,29.834532,2015-10-31,2015-10-30
7,2015-10-31,8,Health Care,INCY,1.650718,0.05,117.410000,117.530000,2015-10-31,2015-10-30
8,2015-10-31,9,Utilities,TE-201606,1.487465,0.05,26.455729,26.328960,2015-10-31,2015-10-30
9,2015-10-31,10,Consumer Discretionary,AMZN,1.481530,0.05,31.300500,31.295002,2015-10-31,2015-10-30


In [6]:
# Simulacion de ejecucion con reglas estrictas de costes y delistings
INITIAL_CASH = 250_000.0
FEE_RATE = 0.0023
FEE_MIN = 23.0
EPS = 1e-9


def trade_fee(value):
    return max(FEE_RATE * value, FEE_MIN)


holdings = {}  # symbol -> shares
cash = INITIAL_CASH
history = []
orders = []

rebalance_info = (
    selection[['rebalance_date', 'trade_date']]
    .drop_duplicates()
    .sort_values('rebalance_date')
    .itertuples(index=False)
)

prev_trade_d = None

for row_info in rebalance_info:
    rebalance_d = pd.Timestamp(row_info.rebalance_date)
    trade_d = pd.Timestamp(row_info.trade_date)

    day_sel = selection[selection['rebalance_date'] == rebalance_d].copy()
    target_symbols = set(day_sel['symbol'])
    target_weights = day_sel.set_index('symbol')['weight'].to_dict()
    sector_map_today = day_sel.set_index('symbol')['sector'].to_dict()

    # Regla de delisting: si deja de cotizar entre rebalanceos, vender a CLOSE del dia de salida.
    if prev_trade_d is not None:
        for sym in list(holdings.keys()):
            px_sym = prices_by_symbol.get(sym)
            if px_sym is None:
                continue

            window = px_sym.loc[(px_sym.index > prev_trade_d) & (px_sym.index <= trade_d)]
            if window.empty:
                continue

            # Si no hay cotizacion en el trade day del rebalanceo, asumimos salida antes de trade_d.
            if trade_d not in window.index:
                exit_date = window.index.max()
                exit_px = float(window.loc[exit_date, 'close'])
                value = holdings[sym] * exit_px
                fee = trade_fee(value)
                cash += value - fee
                orders.append({
                    'date': exit_date,
                    'rebalance_date': rebalance_d,
                    'trade_date': trade_d,
                    'symbol': sym,
                    'sector': sector_map_today.get(sym),
                    'side': 'SELL_DELIST',
                    'price': exit_px,
                    'value': value,
                    'fee': fee,
                })
                del holdings[sym]

    # Valor de cartera a OPEN del trade day
    port_value_open = cash
    day_open = {}
    day_close = {}

    for sym, sh in holdings.items():
        px_sym = prices_by_symbol.get(sym)
        if px_sym is None:
            continue

        if trade_d in px_sym.index:
            px_o = float(px_sym.loc[trade_d, 'open'])
            px_c = float(px_sym.loc[trade_d, 'close'])
            day_open[sym] = px_o
            day_close[sym] = px_c
            port_value_open += sh * px_o
        else:
            # fallback de valoracion al ultimo close disponible
            last_rows = px_sym.loc[px_sym.index <= trade_d]
            if not last_rows.empty:
                port_value_open += sh * float(last_rows.iloc[-1]['close'])

    # Objetivo en USD por activo seleccionado (20 x 5%)
    target_values = {sym: port_value_open * target_weights[sym] for sym in target_symbols}

    # 1) Ventas a OPEN (salidas y rebalanceo a la baja)
    for sym in list(holdings.keys()):
        if sym not in day_open:
            continue

        current_value = holdings[sym] * day_open[sym]
        target_value = target_values.get(sym, 0.0)
        if current_value > target_value + EPS:
            sell_value = current_value - target_value
            sell_shares = sell_value / day_open[sym]
            fee = trade_fee(sell_value)
            cash += sell_value - fee
            holdings[sym] -= sell_shares
            orders.append({
                'date': trade_d,
                'rebalance_date': rebalance_d,
                'trade_date': trade_d,
                'symbol': sym,
                'sector': sector_map_today.get(sym),
                'side': 'SELL',
                'price': day_open[sym],
                'value': sell_value,
                'fee': fee,
            })
            if holdings[sym] <= EPS:
                del holdings[sym]

    # Completar precios CLOSE para seleccionados
    for sym in target_symbols:
        if sym in day_close:
            continue
        px_sym = prices_by_symbol.get(sym)
        if px_sym is not None and trade_d in px_sym.index:
            day_close[sym] = float(px_sym.loc[trade_d, 'close'])

    # 2) Compras a CLOSE (entradas y rebalanceo al alza)
    for _, row_sel in day_sel.iterrows():
        sym = row_sel['symbol']
        px_close = day_close.get(sym)
        if px_close is None or px_close <= 0:
            continue

        target_value = target_values.get(sym, 0.0)
        current_value = holdings.get(sym, 0.0) * px_close
        if current_value + EPS < target_value:
            buy_value = target_value - current_value
            fee = trade_fee(buy_value)
            if cash >= buy_value + fee:
                buy_shares = buy_value / px_close
                cash -= buy_value + fee
                holdings[sym] = holdings.get(sym, 0.0) + buy_shares
                orders.append({
                    'date': trade_d,
                    'rebalance_date': rebalance_d,
                    'trade_date': trade_d,
                    'symbol': sym,
                    'sector': row_sel['sector'],
                    'side': 'BUY',
                    'price': px_close,
                    'value': buy_value,
                    'fee': fee,
                })

    # Valor de cartera al cierre del trade day
    close_value = cash
    for sym, sh in holdings.items():
        px_sym = prices_by_symbol.get(sym)
        if px_sym is None:
            continue
        if trade_d in px_sym.index:
            close_value += sh * float(px_sym.loc[trade_d, 'close'])
        else:
            last_rows = px_sym.loc[px_sym.index <= trade_d]
            if not last_rows.empty:
                close_value += sh * float(last_rows.iloc[-1]['close'])

    history.append({
        'date': rebalance_d,
        'trade_date': trade_d,
        'portfolio_value': close_value,
        'cash': cash,
        'num_positions': len(holdings),
    })

    prev_trade_d = trade_d

history_df = pd.DataFrame(history)
orders_df = pd.DataFrame(orders)
history_df.head()


,date,trade_date,portfolio_value,cash,num_positions
0,2015-10-31,2015-10-30,249453.750000,11953.750000,19
1,2015-11-30,2015-11-30,258267.110664,9867.560888,19
2,2015-12-31,2015-12-31,260845.708783,10850.268136,19
3,2016-01-31,2016-01-29,232404.274775,526.388645,20
4,2016-02-29,2016-02-29,228352.233836,10396.202382,19


In [8]:
# Coste de transaccion por rebalanceo
fees_by_rebalance = (
    orders_df.groupby('rebalance_date', as_index=False)['fee']
             .sum()
             .rename(columns={'rebalance_date': 'date', 'fee': 'txn_cost'})
)

print('Coste total de transaccion:', fees_by_rebalance['txn_cost'].sum())
print('Ordenes ejecutadas:', len(orders_df))
print('Valor final cartera:', history_df['portfolio_value'].iloc[-1] if len(history_df) else np.nan)
print('Posiciones por rebalanceo (min/max):', history_df['num_positions'].min(), '/', history_df['num_positions'].max())

fees_by_rebalance.head()

Coste total de transaccion: 364614.7694822704
Ordenes ejecutadas: 3424
Valor final cartera: 8156445.178268447
Posiciones por rebalanceo (min/max): 19 / 20


,date,txn_cost
0,2015-10-31,546.250000
1,2015-11-30,740.271418
2,2015-12-31,889.897530
3,2016-01-31,552.213954
4,2016-02-29,800.728091
